# Decision Tree Implementation

## Overview

In this notebook, I applied **Decision Tree** to predict Big Five personality traits (Extraversion, Agreeableness, Conscientiousness, Neuroticism, Openness) from textual life narrative responses (Q1–Q32).

## What are Decision and Regression Trees?
A **Decision Tree** partitions the feature space by repeatedly asking yes/no questions (splits) about features, creating a tree of decisions. For regression (continuous targets like Big Five scores), it's called a **Regression Tree** — each leaf predicts the **mean** of training samples that fall into that region.

### Splitting criterion
At each node, the tree finds the feature and threshold that minimizes **Mean Squared Error (MSE)** across the two resulting groups.

### Why Decision Trees for personality prediction?
- Highly interpretable — you can visualize exactly what splits are made
- Captures non-linear relationships naturally
- Useful baseline before ensemble methods (Random Forests, Gradient Boosting)
- Feature importance is straightforward to compute

### Key risk: Overfitting
Fully grown trees memorize training data. We control this with `max_depth` and `min_samples_split`.

### Pipeline
```
Narrative text (Q columns)
        ↓
TF-IDF Vectorization
        ↓
TruncatedSVD → 100 dense dimensions
        ↓
DecisionTreeRegressor (one per trait)
        ↓
5-Fold Cross-Validation + Evaluation
```

## 1. Data preprocessing, import required packages

As specified in the random forest module, all narrative columns are combined into a single text string per participant, then apply **TF-IDF** (Term Frequency–Inverse Document Frequency) to convert text into a numeric matrix. 

The key process of supervised learning is for model to learn patterns from training set and evaluate model performance on unseen test set. Thus, data is split into **80% training** and **20% testing**.`random_state=42` ensures reproducibility

In [2]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import sys
import os

sys.path.append(os.path.abspath("../../src"))

from ml_models.decision_tree import DecisionTreePersonality

import warnings
warnings.filterwarnings('ignore')

print("✓ Imports complete")

# Find repo root automatically
repo_root = Path.cwd().resolve()
while not (repo_root / "BFI_2_life_narative_metadata.json").exists():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Could not find BFI_2_life_narative_metadata.json")
    repo_root = repo_root.parent

# Load data
with open(repo_root / "BFI_2_life_narative_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

with open(repo_root / "BFI_2_life_narrative.json", "r", encoding="utf-8") as f:
    responses = json.load(f)

df = pd.DataFrame(responses)

# Reverse-code BFI items
reverse_items = metadata["reverse_code_items"]

for item in reverse_items:
    if item in df.columns:
        df[item] = 6 - df[item]

# Big Five trait means
traits = {
    key: value
    for key, value in metadata.items()
    if (
        isinstance(value, list)
        and value
        and isinstance(value[0], str)
        and value[0].startswith("Item")
        and not key.startswith(("Item", "Q", "CWB", "OCB"))
        and key != "reverse_code_items"
    )
}

for trait, items in traits.items():
    available_items = [item for item in items if item in df.columns]
    df[trait] = df[available_items].mean(axis=1)

# CWB and OCB means
cwb_cols = [f"CWB{i}" for i in range(1, 11) if f"CWB{i}" in df.columns]
ocb_cols = [f"OCB{i}" for i in range(1, 11) if f"OCB{i}" in df.columns]

df["CWB"] = df[cwb_cols].mean(axis=1)
df["OCB"] = df[ocb_cols].mean(axis=1)

# Text features
q_cols = [c for c in df.columns if isinstance(c, str) and c.startswith("Q")]

X_text = df[q_cols].copy()
X_all = X_text.fillna("").agg(" ".join, axis=1)

# Targets
big_five = [
    "Extraversion",
    "Agreeableness",
    "Conscientiousness",
    "Neuroticism",
    "Openness",
]

y = df[big_five].copy()

print("✓ Data preprocessing complete")
print("Repo root:", repo_root)
print("df shape:", df.shape)
print("X_text shape:", X_text.shape)
print("X_all shape:", X_all.shape)
print(y.describe().round(3))

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=42
)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")

✓ Imports complete
✓ Data preprocessing complete
Repo root: /Users/cindy/cmor438_Spring2026/cmor438_Spring2026
df shape: (500, 138)
X_text shape: (500, 32)
X_all shape: (500,)
       Extraversion  Agreeableness  Conscientiousness  Neuroticism  Openness
count       500.000        500.000            500.000      500.000   500.000
mean          3.177          3.786              3.642        2.879     3.833
std           0.761          0.576              0.710        0.846     0.618
min           1.000          2.000              1.083        1.000     1.750
25%           2.667          3.417              3.167        2.333     3.417
50%           3.167          3.833              3.750        2.917     3.833
75%           3.750          4.167              4.083        3.417     4.250
max           5.000          5.000              5.000        4.917     5.000
Training samples : 400
Test samples     : 100


## 2. Train & test model & Cross validation

**5-fold cross-validation** is applied to give a more reliable estimate for model performance.

`max_depth` is the most important hyperparameter:
- `max_depth=None` → fully grown tree → overfits badly
- `max_depth=3–6` → shallow tree → better generalization

`min_samples_split` is also tuned to further control overfitting.

In [3]:
tree_model = DecisionTreePersonality()
tree_model.fit(X_train, y_train)

tree_results = tree_model.evaluate(X_test, y_test)
tree_results

,R²,MAE,Pearson r
Trait,,,
Extraversion,-0.6888,0.7944,0.1277
Agreeableness,-1.6506,0.6566,-0.1007
Conscientiousness,-0.4783,0.7038,0.1972
Neuroticism,-0.5712,0.9233,0.0913
Openness,-0.7775,0.6984,0.0483


In [4]:
tree_cv_results = tree_model.cross_validate(X_all, y, cv=5)
tree_cv_results

,CV Mean R²,CV Std R²,CV MAE,Pearson r
Trait,,,,
Extraversion,-0.8270,0.1579,0.8305,0.1297
Agreeableness,-1.0604,0.3368,0.6495,-0.0353
Conscientiousness,-0.9248,0.2280,0.7669,0.0737
Neuroticism,-0.7311,0.1396,0.8680,0.1024
Openness,-0.9561,0.2593,0.6894,0.0452


## Results analysis

**Decision Tree** produced the weakest results across all models applied in predicting personality with narrative data. It had the lowest Pearson correlations for most traits, including near-zero or negative relationships for **Agreeableness** (*r* = -0.0353), indicating that predictions were no better than random alignment with observed scores for those traits. Its R² values were also negative across every trait, meaning the model performed worse than simply predicting the sample mean. In addition, Decision Tree had the highest or near-highest MAE values across traits, showing larger average prediction errors. These findings suggest that a single tree likely lacked the flexibility and stability needed to capture complex high-dimensional and sparse patterns like personality and may have been highly sensitive to sample-specific noise.